# RQ1: matched shifts in the shared Gemma residual stream

Run this notebook only after `03_ood_em_baseline.ipynb` has produced a **passed, cryptographically bound three-seed OOD gate**. This is the project's matched-token `M_ft − M_base` extension. It is not the paper's final-token/within-space SVD geometry reproduction and it does not establish a causal vision-tower origin.

Each primary seed uses at least 50 unique matched prompt/image pairs, reviewed EM and control prompt banks, paired prompt-level bootstrap resampling, and layers 20 and 32 with zero-based indexing.

## 1. Confirm A100, mount Drive, and select the seed

In [ ]:
from pathlib import Path
import os
import torch

assert torch.cuda.is_available(), 'Enable a GPU runtime.'
GPU_NAME = torch.cuda.get_device_name(0)
assert 'A100' in GPU_NAME and torch.cuda.is_bf16_supported(), GPU_NAME

SEED = 42  # repeat with 43 and 44
assert SEED in {42, 43, 44}
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_PROJECT = Path('/content/drive/MyDrive/em-displacement-vlm')
os.environ['EM_DATA_DIR'] = str(DRIVE_PROJECT / 'data')
os.environ['EM_CHECKPOINT_DIR'] = str(DRIVE_PROJECT / 'checkpoints')
os.environ['EM_RESULTS_DIR'] = str(DRIVE_PROJECT / 'results')
os.environ['HF_HOME'] = '/content/hf-cache'
print('GPU:', GPU_NAME, 'seed:', SEED)

## 2. Use a clean, versioned checkout and install the extraction runtime

In [ ]:
import subprocess
import sys

REPO_URL = 'https://github.com/rlogger/em-displacement-vlm.git'
REPO_DIR = Path('/content/em-displacement-vlm')
if REPO_DIR.exists():
    assert (REPO_DIR / '.git').is_dir(), f'{REPO_DIR} is not a git clone; restart runtime.'
    assert not subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip(), 'Clone is dirty; restart runtime.'
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', '--prune', 'origin', 'main'])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'checkout', '--detach', 'origin/main'])
else:
    subprocess.check_call(['git', 'clone', '--branch', 'main', '--single-branch', REPO_URL, str(REPO_DIR)])
%cd {REPO_DIR}
REPO_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
constraints_path = REPO_DIR / 'constraints' / 'colab.txt'
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    '--constraint', str(constraints_path), 'unsloth', 'datasets>=2.19',
    'huggingface-hub>=0.23', 'safetensors>=0.4', 'pyyaml>=6.0', 'peft',
])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR), '--no-deps'])
subprocess.check_call([sys.executable, '-c', "import torch, unsloth, safetensors; print(torch.__version__, torch.version.cuda, 'runtime OK')"])
print('Repository commit:', REPO_COMMIT)

## 3. Verify the OOD gate, local adapter, split, and reviewed prompt banks

Prepare two immutable JSON/JSONL banks before looking at activations: at least 50 EM-relevant prompts and 50 non-overlapping matched controls. Each review metadata JSON must contain `review_status: approved`, the manifest SHA-256, reviewer, date, and selection policy.

In [ ]:
import hashlib
import json

SPLIT_ROOT = DRIVE_PROJECT / 'data' / 'splits' / f'seed{SEED}'
ADAPTER_DIR = DRIVE_PROJECT / 'checkpoints' / f'FT_R32_gemma3_faces_seed{SEED}'
OOD_GATE = DRIVE_PROJECT / 'results' / 'ood' / 'ood_three_seed_gate.json'
PROBE_ROOT = DRIVE_PROJECT / 'data' / 'rq1_probes'
EM_PROMPTS = PROBE_ROOT / 'em_primary_v1.jsonl'
EM_PROMPTS_REVIEW = PROBE_ROOT / 'em_primary_v1.review.json'
CONTROL_PROMPTS = PROBE_ROOT / 'control_v1.jsonl'
CONTROL_PROMPTS_REVIEW = PROBE_ROOT / 'control_v1.review.json'

for path in (
    SPLIT_ROOT / 'manifest.json', ADAPTER_DIR / 'adapter_config.json',
    ADAPTER_DIR / 'run_metadata.json', ADAPTER_DIR / 'reproduction_manifest.json',
    OOD_GATE, EM_PROMPTS, EM_PROMPTS_REVIEW, CONTROL_PROMPTS, CONTROL_PROMPTS_REVIEW,
):
    assert path.is_file(), f'Missing required artifact: {path}'
gate = json.loads(OOD_GATE.read_text())
assert gate.get('behavioral_gate') == 'pass', 'Primary RQ1 requires a passed OOD gate.'
assert gate.get('seed_coverage') == [42, 43, 44], gate

def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

for manifest, review in ((EM_PROMPTS, EM_PROMPTS_REVIEW), (CONTROL_PROMPTS, CONTROL_PROMPTS_REVIEW)):
    metadata = json.loads(review.read_text())
    assert metadata.get('review_status') == 'approved'
    assert metadata.get('manifest_sha256') == sha256(manifest)
    assert all(str(metadata.get(field, '')).strip() for field in ('reviewed_by', 'reviewed_at', 'selection_policy'))
print('Primary readiness artifacts are present; the extractor will revalidate all hashes.')

## 4. Materialize the immutable primary RQ1 config

In [ ]:
import yaml

N_MATCHED_PAIRS = 50
RQ1_OUTPUT_DIR = DRIVE_PROJECT / 'results' / 'rq1' / f'seed{SEED}'
rq1_cfg = yaml.safe_load(Path('configs/extract_rq1.yaml').read_text())
rq1_cfg.update({
    'analysis_tier': 'primary', 'run_name': f'extract_rq1_primary_seed{SEED}',
    'seed': SEED, 'ft_adapter': str(ADAPTER_DIR), 'split_root': str(SPLIT_ROOT),
    'output_dir': str(RQ1_OUTPUT_DIR), 'ood_gate_manifest': str(OOD_GATE),
    'review_summary': None, 'review_provenance': None,
    'n_text_prompts': N_MATCHED_PAIRS, 'n_multimodal_prompts': N_MATCHED_PAIRS,
    'text_probe_manifest': str(EM_PROMPTS),
    'text_probe_manifest_sha256': sha256(EM_PROMPTS),
    'text_probe_review_metadata': str(EM_PROMPTS_REVIEW),
    'control_prompt_manifest': str(CONTROL_PROMPTS),
    'control_prompt_manifest_sha256': sha256(CONTROL_PROMPTS),
    'control_prompt_review_metadata': str(CONTROL_PROMPTS_REVIEW),
    'language_layers': [20, 32], 'bootstrap_samples': 10000,
    'null_samples': 10000, 'load_in_4bit': False,
})
RQ1_CONFIG = DRIVE_PROJECT / 'runs' / f'extract_rq1_primary_seed{SEED}.yaml'
rendered = yaml.safe_dump(rq1_cfg, sort_keys=False)
if RQ1_CONFIG.exists() and RQ1_CONFIG.read_text() != rendered:
    raise RuntimeError(f'Existing primary config differs: {RQ1_CONFIG}')
if not RQ1_CONFIG.exists():
    RQ1_CONFIG.write_text(rendered)
print(RQ1_CONFIG.read_text())

## 5. Extract and seal this seed's geometry bundle

In [ ]:
subprocess.check_call([sys.executable, 'scripts/extract_rq1.py', '--config', str(RQ1_CONFIG)])
RQ1_BUNDLE = RQ1_OUTPUT_DIR / 'rq1_geometry.json'
RQ1_SIDECAR = RQ1_OUTPUT_DIR / 'rq1_geometry.meta.json'
ACTIVATIONS = RQ1_OUTPUT_DIR / 'activation_matrices.safetensors'
assert all(path.is_file() for path in (RQ1_BUNDLE, RQ1_SIDECAR, ACTIVATIONS))
bundle = json.loads(RQ1_BUNDLE.read_text())
assert bundle['analysis_tier'] == 'primary'
assert bundle['analysis_method'] == 'matched_token_ft_shift_extension_v1'
assert bundle['protocol']['n_pairs'] >= 50
assert bundle['protocol']['capture']['bootstrap_unit'] == 'matched_prompt_image_pair'
for condition in ('geometry', 'control_geometry'):
    assert bundle.get(condition), f'Missing {condition}'
    for layer_stats in bundle[condition].values():
        assert layer_stats['image_token_counts']['min'] == 256
        assert layer_stats['n_text'] == layer_stats['n_image_token']
print('Sealed primary seed bundle:', RQ1_BUNDLE)

## 6. After all three extractions, run the strict aggregation

Repeat sections 1–5 for seeds 43 and 44. The aggregate command rejects missing seeds, incompatible protocol fingerprints, missing controls, weak provenance, fewer than 50 pairs, or altered bundles. Never use `--allow-legacy` for a research claim.

In [ ]:
RQ1_BUNDLES = [
    DRIVE_PROJECT / 'results' / 'rq1' / f'seed{seed}' / 'rq1_geometry.json'
    for seed in (42, 43, 44)
]
assert all(path.is_file() for path in RQ1_BUNDLES), RQ1_BUNDLES
RQ1_SUMMARY = DRIVE_PROJECT / 'results' / 'rq1' / 'rq1_three_seed_summary.json'
subprocess.check_call([
    sys.executable, 'scripts/aggregate_rq1.py', *map(str, RQ1_BUNDLES),
    '--out', str(RQ1_SUMMARY),
])
summary = json.loads(RQ1_SUMMARY.read_text())
print(json.dumps(summary, indent=2))

## Interpretation boundary

- `consistent_positive_alignment` requires positive observed cosines and positive lower bounds of the paired bootstrap interval in all three seeds.
- The equal-norm random-direction tail fraction is a descriptive orientation reference, **not a p-value and not part of the decision rule**.
- A positive result supports shared-residual alignment under this registered extension. It does not prove a shared causal mechanism, a raw vision-tower origin, intervention efficacy, or displacement.
- Preserve mixed, negative, and imprecise outcomes. Do not omit a seed or tune prompts/layers after seeing the result.